## IMPORTING THE LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

import joblib

## MINE IS NOT IN ROOT DIRECTORY IF URS IS THERE DO THERE

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "src":
    ROOT = ROOT.parent              

DATA_PATH = ROOT / "data" / "sensor.csv"
MODEL_DIR = ROOT / "model"
MODEL_DIR.mkdir(exist_ok=True)

print("File exists:", DATA_PATH.exists())   

File exists: True


## SIMPLE READING DATASET AND GETTING THE IDEA OF SHAPE SIZE DESCRIBE AND HEAD

In [29]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df["machine_status"].value_counts())

Shape: (220320, 55)
machine_status
NORMAL        205836
RECOVERING     14477
BROKEN             7
Name: count, dtype: int64


## REMOVING THE DUPLICATE BALUE AND SETTING UP THRESHOLD VALUE ACCORDING TO THE PROJECT

In [ ]:
df = df.dropna(axis=1, thresh=int(len(df) * 0.5))   
for c in ["timestamp", "Unnamed: 0"]:
    if c in df.columns:
        df = df.drop(columns=c)                     
df = df.ffill().bfill()                             
print("Remaining NaNs:", df.isna().sum().sum())     

Remaining NaNs: 0


## SPLITTING INTO TRANING AND TESTING THE DATA

In [ ]:
X = df.drop(columns="machine_status").astype("float32")   
y = (df["machine_status"] != "NORMAL").astype(int)         

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

scaler = StandardScaler().fit(X_train)            
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print("Test anomalies:", int(y_test.sum()), "/", len(y_test))

Test anomalies: 2897 / 44064


## HERE WE USE SUPERVISIED MODELS 

In [ ]:
supervised = {
    "LogisticRegression":   LogisticRegression(max_iter=1000, class_weight="balanced"),
    "DecisionTree":         DecisionTreeClassifier(max_depth=10, class_weight="balanced", random_state=42),
    "KNN":                  KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    "RandomForest":         RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                                   random_state=42, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

print(f"{'Model':22s}{'P':>7s}{'R':>7s}{'F1':>7s}{'Time':>8s}")
for name, mdl in supervised.items():
    t = time.time()
    mdl.fit(X_train_s, y_train)
    pred = mdl.predict(X_test_s)
    p = precision_score(y_test, pred, zero_division = 0)
    r = recall_score(y_test, pred, zero_division = 0)
    f = f1_score(y_test, pred, zero_division = 0)
    print(f"{name:22s}{p:7.3f}{r:7.3f}{f:7.3f}{time.time()-t:6.1f}s")

Model                       P      R     F1    Time
LogisticRegression      0.952  0.999  0.975   1.7s
DecisionTree            0.978  0.999  0.989   4.2s
KNN                     0.997  0.997  0.997  15.9s
RandomForest            0.999  1.000  1.000  10.8s
HistGradientBoosting    0.999  1.000  1.000   3.1s


## KEEP IN MIDN THINGS BREAK AT 6.5 THRESHOLD

In [ ]:
X_normal = X_train_s[y_train.values == 0]     
rate = float(y_train.mean())                  

sub = np.random.RandomState(42).choice(len(X_normal), 20000, replace=False)
X_sub = X_normal[sub]

def evaluate(name, train_scores, test_scores):
    line = np.quantile(train_scores, 1 - rate)
    pred = (test_scores >= line).astype(int)
    p = precision_score(y_test, pred, zero_division = 0)
    r = recall_score(y_test, pred, zero_division = 0)
    f = f1_score(y_test, pred, zero_division = 0)
    print(f"{name:22s}{p:7.3f}{r:7.3f}{f:7.3f}")
    return f, pred

print(f"{'Model':22s}{'P':>7s}{'R':>7s}{'F1':>7s}")
unsup = {}

iso = IsolationForest(n_estimators = 200, random_state = 42, n_jobs = -1).fit(X_normal)
unsup["IsolationForest"] = (iso, *evaluate("IsolationForest",
    -iso.decision_function(X_normal), -iso.decision_function(X_test_s)))

lof = LocalOutlierFactor(n_neighbors = 20, novelty = True, n_jobs = -1).fit(X_sub)
unsup["LocalOutlierFactor"] = (lof, *evaluate("LocalOutlierFactor",
    -lof.decision_function(X_sub), -lof.decision_function(X_test_s)))

ee = EllipticEnvelope(contamination = rate, support_fraction = 0.9, random_state = 42).fit(X_sub)
unsup["EllipticEnvelope"] = (ee, *evaluate("EllipticEnvelope",
    -ee.decision_function(X_sub), -ee.decision_function(X_test_s)))

Model                       P      R     F1
IsolationForest         0.518  0.994  0.681
LocalOutlierFactor      0.437  0.844  0.576
EllipticEnvelope        0.507  1.000  0.673


## CHOOSING THE BEST MODEL

In [ ]:
best_name = max(unsup, key=lambda n: unsup[n][1])   
best_model = unsup[best_name][0]
threshold = float(np.quantile(-iso.decision_function(X_normal), 1 - rate))

joblib.dump(
    {"model": best_model, "scaler": scaler, "threshold": threshold,
     "feature_names": list(X.columns), "name": best_name},
    MODEL_DIR / "sensorguard.joblib"
)
print("Saved:", best_name, "->", MODEL_DIR / "sensorguard.joblib")

Saved: IsolationForest -> c:\Users\yk036\Desktop\SensorGuard Predictive Maintenance Anomaly Detection API\model\sensorguard.joblib
